In [1]:
import requests
import urllib.request
import pandas as pd
import time
import datetime
from datetime import date
from persiantools.jdatetime import JalaliDate
import json

In [2]:
# تابع تولید کننده تاریخ شمسی از روی تاریخ میلادی
def miladi_to_shamsi (date_miladi):
    miladi_list = list(str(date_miladi))
    year = int(miladi_list[0] + miladi_list[1] + miladi_list[2] + miladi_list[3])
    mounth = int(miladi_list[4] + miladi_list[5])
    day = int(miladi_list[6] + miladi_list[7])
    date_shamsi = str(JalaliDate.to_jalali(year , mounth , day))
    date_shamsi = date_shamsi.split('-')
    date_shamsi = date_shamsi[0] + date_shamsi[1] + date_shamsi[2]
    return date_shamsi

# استخراج داده های تعدیل شده برای نمادها بر اساس آیدی کد هر شرکت
# فقط روزهای معامله شده
def chart_data(id_code):
    tex = requests.get(r'https://members.tsetmc.com/tsev2/chart/data/Financial.aspx?i='+ str(id_code) + '&t=ph&a=1' , timeout=10).text.split(';')
    list_rows = []
    [list_rows.append(row.split(',')) for row in tex]
    df_chart_data = pd.DataFrame(list_rows , columns = ['Miladi_date' , 'High' , 'Low' , 'Open' , 'Close' , 'Volume' , 'End'])
    end_price = df_chart_data['End']
    aa = end_price[0]
    yes_price = end_price.shift(1)
    yes_price[0] = aa
    df_chart_data['Yes_price'] = yes_price
    list_shamsi_date = []
    for miladi_date in list(df_chart_data['Miladi_date']):
        shamsi_date = miladi_to_shamsi(miladi_date)
        list_shamsi_date.append(int(shamsi_date))
    df_chart_data['Shamsi_date'] = list_shamsi_date
    df_chart_data['Id_code']=[id_code]*len(df_chart_data)
        
    return df_chart_data


# استخراج داده های تعدیل نشده ( اورجینال ) بر اساس ایدی کد هر شرکت
# تمام روزها
def orginal_data(id_code):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/112.0'}
    tex = requests.get(r'https://cdn.tsetmc.com/api/ClosingPrice/GetClosingPriceDailyList/'+ str(id_code) + '/0',
                       headers = headers , timeout=10).json()

    df_orginal = pd.DataFrame(tex['closingPriceDaily'])

    df_orginal_data = df_orginal.get(['dEven' , 'priceMax' , 'priceMin' , 'pClosing' , 'pDrCotVal' , 'priceFirst' , 
                                      'priceYesterday' , 'qTotCap' , 'qTotTran5J' , 'zTotTran'])
    df_orginal_data.columns = ['O_Miladi_date' , 'O_High' , 'O_Low' , 'O_End', 'O_Close' , 'O_Open' 
                               , 'O_Yes_price' ,'O_Value', 'O_Volume' , 'O_Count' ]

    #df_orginal_data = df_orginal_data[df_orginal_data['O_Volume']!=0]
    df_orginal_data = df_orginal_data[df_orginal_data['O_Miladi_date']!= '']
    list_shamsi_date = []
    for miladi_date in list(df_orginal_data['O_Miladi_date']):
        shamsi_date = miladi_to_shamsi(miladi_date)
        list_shamsi_date.append(int(shamsi_date))
    df_orginal_data['O_Shamsi_date'] = list_shamsi_date
    df_orginal_data['Id_code']=[id_code]*len(df_orginal_data)
        
    return df_orginal_data

In [ ]:
# داده تعدیل نشده
# برای شاخص قیمت (وزنی-ارزشی): 5798407779416661
# شاخصص هم وزن: 67130298613737946
# شاخص قیمت هم وزن: 8384385859414435
df_chart_data = chart_data(32097828799138957)
df_chart_data


,Miladi_date,High,Low,Open,Close,Volume,End,Yes_price,Shamsi_date,Id_code
0,20080629,11,11,11,11,189389047,11,11,13870409,7745894403636165
1,20080630,11,11,11,11,19643005,11,11,13870410,7745894403636165
2,20080701,11,11,11,11,22646072,11,11,13870411,7745894403636165
3,20080702,11,11,11,11,3591428,11,11,13870412,7745894403636165
4,20080705,11,11,11,11,7310389,11,11,13870415,7745894403636165
...,...,...,...,...,...,...,...,...,...,...
3374,20260128,7080,6800,6800,6890,947198634,6820,7010,14041108,7745894403636165
3375,20260131,6960,6620,6710,6620,308310751,6650,6820,14041111,7745894403636165
3376,20260201,6840,6480,6560,6840,294671555,6720,6650,14041112,7745894403636165
3377,20260202,6920,6900,6920,6920,186748566,6920,6720,14041113,7745894403636165


In [ ]:
df_orginal_data = orginal_data(32097828799138957)
df_orginal_data

,O_Miladi_date,O_High,O_Low,O_End,O_Close,O_Open,O_Yes_price,O_Value,O_Volume,O_Count,O_Shamsi_date,Id_code
0,20260203,7110.0,6780.0,6920.0,6990.0,7110.0,6920.0,1.482724e+12,214356317.0,8298.0,14041114,7745894403636165
1,20260202,6920.0,6900.0,6920.0,6920.0,6920.0,6720.0,1.292266e+12,186748566.0,4705.0,14041113,7745894403636165
2,20260201,6840.0,6480.0,6720.0,6840.0,6560.0,6650.0,1.979187e+12,294671555.0,10116.0,14041112,7745894403636165
3,20260131,6960.0,6620.0,6650.0,6620.0,6710.0,6820.0,2.049531e+12,308310751.0,7367.0,14041111,7745894403636165
4,20260128,7080.0,6800.0,6820.0,6890.0,6800.0,7010.0,6.458560e+12,947198634.0,25112.0,14041108,7745894403636165
...,...,...,...,...,...,...,...,...,...,...,...,...
4200,20080705,6110.0,5952.0,6076.0,6076.0,6070.0,5933.0,4.442506e+10,7310389.0,1066.0,13870415,7745894403636165
4201,20080702,5982.0,5808.0,5933.0,5933.0,5869.0,5808.0,2.130842e+10,3591428.0,551.0,13870412,7745894403636165
4202,20080701,5839.0,5790.0,5808.0,5808.0,5829.0,5948.0,1.315672e+11,22646072.0,805.0,13870411,7745894403636165
4203,20080630,5974.0,5801.0,5948.0,5948.0,5974.0,5800.0,1.168543e+11,19643005.0,2021.0,13870410,7745894403636165
